In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ============================================================
# Parameters
# ============================================================

g = 9.80665
rho = 3.0                         # Display density
ppi = 160.0 * rho
C = g * 39.37 * ppi * 0.84        # Physical coefficient

mu = 0.015                        # Fling friction
beta = 0.35                       # Inflexion factor
r = np.log(0.78) / np.log(0.9)    # Deceleration rate

S = 0.5                           # Start tension
E = 1.0                           # End tension

P1 = S * beta
P2 = 1.0 - E * (1.0 - beta)

print(f"ppi = {ppi:.1f}")
print(f"C = {C:.2f}")
print(f"r = {r:.6f}")
print(f"Bezier control points: (0, 0), ({P1:.3f}, {S:.3f}), ({P2:.3f}, 1), (1, 1)")


# ============================================================
# Fling scale: distance and duration from launch velocity
# ============================================================

def fling_distance(v0):
    """
    Total unsigned fling distance D(v0) in pixels.
    v0 is the launch velocity in px/s.
    """
    v = np.abs(v0)
    return mu * C * (beta * v / (mu * C)) ** (r / (r - 1.0))


def fling_duration(v0):
    """
    Total fling duration T(v0) in milliseconds.
    v0 is the launch velocity in px/s.
    """
    v = np.abs(v0)
    return 1000.0 * (beta * v / (mu * C)) ** (1.0 / (r - 1.0))


# Example launch velocity
v0 = 5000.0

D = fling_distance(v0)
T = fling_duration(v0)

print(f"\nExample fling:")
print(f"v0 = {v0:.0f} px/s")
print(f"D  = {D:.2f} px")
print(f"T  = {T:.2f} ms")


# ============================================================
# Cubic Bezier spline
#
# tau(x) = 3x(1-x)[(1-x)P1 + xP2] + x^3
# p(x)   = 3x(1-x)[(1-x)S + x] + x^3
# ============================================================

def tau_of_x(x):
    """Normalized time tau(x)."""
    return 3.0 * x * (1.0 - x) * ((1.0 - x) * P1 + x * P2) + x**3


def p_of_x(x):
    """Normalized travelled distance p(x)."""
    return 3.0 * x * (1.0 - x) * ((1.0 - x) * S + x) + x**3


# Fine sampling of the parametric curve
x = np.linspace(0.0, 1.0, 5000)

tau = tau_of_x(x)
p = p_of_x(x)


# ============================================================
# Plot 1: Fling distance and duration as functions of velocity
# ============================================================

velocities = np.linspace(100.0, 12000.0, 1000)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))

axes[0].plot(
    velocities,
    fling_distance(velocities),
    linewidth=2.5,
    color="tab:blue"
)
axes[0].set_title("Fling Distance")
axes[0].set_xlabel("Launch velocity |v₀| (px/s)")
axes[0].set_ylabel("Total distance D (px)")

axes[1].plot(
    velocities,
    fling_duration(velocities),
    linewidth=2.5,
    color="tab:orange"
)
axes[1].set_title("Fling Duration")
axes[1].set_xlabel("Launch velocity |v₀| (px/s)")
axes[1].set_ylabel("Duration T (ms)")

fig.suptitle("Physical Fling Scale", fontsize=15)
plt.tight_layout()
plt.show()


# ============================================================
# Plot 2: tau(x), p(x), and the position-versus-time spline
# ============================================================

control_x = np.array([0.0, P1, P2, 1.0])
control_y = np.array([0.0, S, 1.0, 1.0])

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: scalar curves over common parameter x
axes[0].plot(x, tau, label=r"$\tau(x)$", linewidth=2.5, color="tab:blue")
axes[0].plot(x, p, label=r"$p(x)$", linewidth=2.5, color="tab:orange")

axes[0].set_title("Two Cubic Bézier Functions")
axes[0].set_xlabel(r"Shared parameter $x$")
axes[0].set_ylabel("Normalized value")
axes[0].set_xlim(0, 1)
axes[0].set_ylim(0, 1.05)
axes[0].legend()

# Right: parametric curve p(tau)
axes[1].plot(
    tau,
    p,
    linewidth=2.8,
    color="tab:blue",
    label=r"Normalized spline $p(\tau)$"
)

axes[1].plot(
    control_x,
    control_y,
    linestyle="--",
    marker="o",
    linewidth=1.4,
    color="tab:orange",
    label="Control polygon"
)

labels = [
    r"$(0, 0)$",
    rf"$({P1:.3f}, {S:.1f})$",
    rf"$({P2:.2f}, 1)$",
    r"$(1, 1)$"
]

for cx, cy, label in zip(control_x, control_y, labels):
    offset_y = 8 if cy < 1 else -15
    axes[1].annotate(
        label,
        (cx, cy),
        xytext=(5, offset_y),
        textcoords="offset points",
        fontsize=9
    )

axes[1].set_title("Normalized Position-versus-Time Spline")
axes[1].set_xlabel(r"Normalized time $\tau$")
axes[1].set_ylabel(r"Normalized distance $p$")
axes[1].set_xlim(0, 1)
axes[1].set_ylim(0, 1.05)
axes[1].set_aspect("equal", adjustable="box")
axes[1].legend(loc="lower right")

plt.tight_layout()
plt.show()


# ============================================================
# Lookup table
#
# The curve is sampled at equally spaced normalized times:
# tau_i = i / N, i = 0, ..., N
# ============================================================

N = 100
tau_table = np.linspace(0.0, 1.0, N + 1)

# Numerically invert tau(x) using monotonic interpolation.
x_table = np.interp(tau_table, tau, x)
p_table = p_of_x(x_table)

# Numerical derivative dp/dtau for velocity lookup.
slope_table = np.gradient(p_table, tau_table)

print("\nFirst ten position values in the lookup table:")
print(np.round(p_table[:10], 6))

print("\nLast ten position values in the lookup table:")
print(np.round(p_table[-10:], 6))


# ============================================================
# Plot 3: Continuous spline and lookup table samples
# ============================================================

plt.figure(figsize=(7.5, 5.5))

plt.plot(
    tau,
    p,
    linewidth=2.6,
    color="tab:blue",
    label="Continuous spline"
)

plt.scatter(
    tau_table,
    p_table,
    s=15,
    color="tab:orange",
    label="Lookup-table entries"
)

plt.title("Normalized Spline Sampled into a 101-Entry Lookup Table")
plt.xlabel(r"Normalized time $\tau$")
plt.ylabel(r"Normalized distance $p$")
plt.xlim(0, 1)
plt.ylim(0, 1.05)
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()


# ============================================================
# Runtime lookup and fling playback
# ============================================================

def lookup_position_and_slope(normalized_time):
    """
    Returns p(tau) and p'(tau) by linear interpolation
    in the sampled lookup table.
    """
    normalized_time = np.clip(normalized_time, 0.0, 1.0)

    position = np.interp(
        normalized_time,
        tau_table,
        p_table
    )

    slope = np.interp(
        normalized_time,
        tau_table,
        slope_table
    )

    return position, slope


def fling_playback(v0, x_start=0.0, number_of_samples=1000):
    """
    Simulates fling position and velocity over time.

    v0: signed launch velocity in px/s
    x_start: start position in px
    """
    signed_distance = np.sign(v0) * fling_distance(v0)
    duration_ms = fling_duration(v0)

    time_ms = np.linspace(0.0, duration_ms, number_of_samples)
    normalized_time = time_ms / duration_ms

    normalized_position, normalized_slope = lookup_position_and_slope(
        normalized_time
    )

    position_px = x_start + normalized_position * signed_distance

    velocity_px_s = (
        normalized_slope
        * signed_distance
        / duration_ms
        * 1000.0
    )

    return time_ms, position_px, velocity_px_s, signed_distance, duration_ms


# ============================================================
# Plot 4: Example fling playback
# ============================================================

time_ms, position_px, velocity_px_s, signed_distance, duration_ms = (
    fling_playback(v0)
)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))

axes[0].plot(
    time_ms,
    position_px,
    linewidth=2.5,
    color="tab:blue"
)
axes[0].set_title("Fling Position Over Time")
axes[0].set_xlabel("Elapsed time t (ms)")
axes[0].set_ylabel("Position x(t) (px)")

axes[1].plot(
    time_ms,
    velocity_px_s,
    linewidth=2.5,
    color="tab:orange"
)
axes[1].set_title("Fling Velocity Over Time")
axes[1].set_xlabel("Elapsed time t (ms)")
axes[1].set_ylabel("Velocity v(t) (px/s)")

fig.suptitle(
    f"Example Fling: v₀ = {v0:.0f} px/s, "
    f"D = {signed_distance:.1f} px, "
    f"T = {duration_ms:.1f} ms",
    fontsize=14
)

plt.tight_layout()
plt.show()


# ============================================================
# Plot 5: Compare several fling velocities
# ============================================================

example_velocities = [1500.0, 3000.0, 5000.0, 8000.0]

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))

for example_v0 in example_velocities:
    time_ms, position_px, velocity_px_s, signed_distance, duration_ms = (
        fling_playback(example_v0)
    )

    axes[0].plot(
        time_ms,
        position_px,
        linewidth=2,
        label=(
            f"v₀={example_v0:.0f} px/s, "
            f"D={signed_distance:.0f} px, "
            f"T={duration_ms:.0f} ms"
        )
    )

    axes[1].plot(
        time_ms / duration_ms,
        position_px / signed_distance,
        linewidth=2,
        label=f"v₀={example_v0:.0f} px/s"
    )

axes[0].set_title("Physical Scale Changes with Launch Velocity")
axes[0].set_xlabel("Elapsed time t (ms)")
axes[0].set_ylabel("Travelled distance (px)")
axes[0].legend(fontsize=8)

axes[1].set_title("All Flings Share the Same Normalized Shape")
axes[1].set_xlabel(r"Normalized time $\tau$")
axes[1].set_ylabel(r"Normalized distance $p$")
axes[1].set_xlim(0, 1)
axes[1].set_ylim(0, 1.05)
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()
```

SyntaxError: invalid syntax (614338454.py, line 385)